# Week 3: Loss Functions, Gradients, and Backpropagation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rrfhwn/neural-architectures-and-representation-learning-course/blob/main/weeks/03/Week_03_Loss_Gradients_Backprop.ipynb)

**Course:** Neural Architectures and Representation Learning (Master level)

## Learning goals

- Understand **loss functions** as a measure of prediction error (MSE, cross-entropy intuition).
- Build intuition for **gradients** and **gradient descent**: move opposite to the direction of steepest ascent.
- Understand **backpropagation** at a conceptual level: gradients flow backward through the network via the chain rule.
- Implement **loss** and **one gradient step** from scratch in NumPy.
- Run a small **training loop** and experiment with different learning rates.

---
## Environment

**Dependencies:** `numpy`, `matplotlib`. Optional: `scikit-learn` for data splitting.

### Local (uv)

From the repo root:

```bash
uv sync
uv run jupyter notebook weeks/03/Week_03_Loss_Gradients_Backprop.ipynb
```

### Colab

1. Open the notebook in Colab via the badge above.
2. Runtime → Run all. NumPy and Matplotlib are pre-installed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Use a simple style for clear plots (works without seaborn)
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    plt.rc('axes', grid=True)
%matplotlib inline

---
## 1. Recap and roadmap

### What we did in Week 2

- **Perceptron:** one neuron, weights $w$, bias $b$; update rule when we misclassify.
- **MLP:** layers of neurons; non-linearity (e.g. ReLU) lets us solve XOR.
- **Training loop:** for each batch: forward pass → loss → backward pass → update parameters.
- **Batch vs online:** we can update on one sample (online), or on a subset (mini-batch), or on the full set (batch).

So far we *used* the training loop without explaining how we obtain the **update direction**. That is the topic of this session.

### Today's roadmap

1. **Loss** — A single scalar measuring prediction error.
2. **Gradients** — The direction of steepest increase; we move in the *opposite* direction.
3. **Gradient descent** — Iterative parameter updates, each proportional to the negative gradient.
4. **Backpropagation** — The chain rule applied backward through each layer to compute all gradients efficiently.
5. **Guided coding** — Implement loss and one gradient step.
6. **Exploration** — Train a small network and study the effect of learning rate.

### Roadmap (visual)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 2))
ax.set_xlim(0, 10)
ax.set_ylim(0, 2)
ax.axis('off')

steps = ['Recap', 'Loss', 'Gradients\n+ descent', 'Backprop', 'Guided\ncoding', 'Exploration']
x_pos = [0.8, 2.2, 3.6, 5.0, 6.4, 7.8]
for i, (s, x) in enumerate(zip(steps, x_pos)):
    ax.add_patch(plt.Rectangle((x, 0.4), 1.0, 1.0, fill=True, facecolor='steelblue', edgecolor='black'))
    ax.text(x + 0.5, 0.9, s, ha='center', va='center', fontsize=8, wrap=True)
    if i < len(x_pos) - 1:
        ax.annotate('', xy=(x_pos[i+1], 0.9), xytext=(x + 1.0, 0.9),
                    arrowprops=dict(arrowstyle='->', color='black'))
ax.set_title('Week 3 flow')
plt.tight_layout()
plt.show()

---
## 2. Loss functions: what we're measuring

### Intuition

The **loss** is a scalar that summarises how far the model's predictions are from the targets on a given batch. Training means *minimising* the loss over the data.

- **Regression:** **mean squared error (MSE)** — average of squared differences between predictions and targets. Larger errors contribute disproportionately more.
- **Classification:** **cross-entropy** (log loss) — measures the divergence between predicted class probabilities and the true label distribution. We focus on MSE in the code today, but the principle is identical.

### Runnable demo: MSE for a few predictions

In [ ]:
# True targets and model predictions (example)
y_true = np.array([1.0, 2.0, 1.5, 3.0])
y_pred = np.array([1.2, 1.8, 1.4, 2.7])

# Mean squared error: average of (pred - target)^2
squared_errors = (y_pred - y_true) ** 2
mse = np.mean(squared_errors)

print("Squared errors per sample:", squared_errors)
print("MSE (average):", mse)

### Pause and reflect

**If one prediction is changed so that it exactly equals its target, what happens to the MSE?**

*Answer:* The MSE decreases — one squared error becomes zero, lowering the average. Each parameter update in training is a small step toward exactly this: reducing individual errors and thereby the overall loss.

### Visualization: loss vs prediction

For a *single* target, we plot how the loss (e.g. squared error) changes as we change the prediction.

In [ ]:
target = 2.0
predictions = np.linspace(0, 4, 200)
losses = (predictions - target) ** 2

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.plot(predictions, losses, 'b-')
ax.axvline(target, color='green', linestyle='--', label=f'target = {target}')
ax.set_xlabel('Prediction')
ax.set_ylabel('Squared error (loss)')
ax.set_title('Loss vs prediction (single target)')
ax.legend()
plt.tight_layout()
plt.show()

The loss is minimised when the prediction equals the target. Training moves the parameters so that predictions shift toward the minimum of this curve.

---
## 3. Gradients and gradient descent (intuition)

### Intuition

The **gradient** of the loss with respect to a parameter indicates the direction in which the loss increases most rapidly. Gradient descent takes a small step in the *opposite* direction at each iteration.

A useful mental image: standing on a hilly landscape, the gradient points uphill. We walk in the opposite direction to reach the valley — the parameter values where the loss is minimised.

### Mental model: one training step

| Step | What happens |
|------|----------------|
| 1. **Forward pass** | Feed inputs through the network → get predictions.
| 2. **Compute loss** | Compare predictions to targets → one scalar (e.g. MSE).
| 3. **Compute gradients** | How would the loss change if we nudged each parameter? (Backprop gives us these.)
| 4. **Update parameters** | Move each parameter a small step *opposite* to its gradient: $w \leftarrow w - \eta \cdot \frac{\partial L}{\partial w}$.

We repeat steps 1–4 for many batches until the loss is small enough.

### Demo: gradient descent on a 2D loss surface

We minimise a simple 2D quadratic (bowl-shaped) function by gradient descent. The contour plot shows the loss surface; the red path is the sequence of parameter updates.

In [ ]:
def loss_2d(w1, w2):
    """Simple bowl: minimum at (1, 1)."""
    return (w1 - 1)**2 + (w2 - 1)**2

def grad_2d(w1, w2):
    """Gradients of loss_2d."""
    return 2*(w1 - 1), 2*(w2 - 1)

# Gradient descent: start at (0, 0), step opposite to gradient
eta = 0.15  # learning rate
w1, w2 = 0.0, 0.0
path = [(w1, w2)]
for _ in range(20):
    g1, g2 = grad_2d(w1, w2)
    w1 = w1 - eta * g1
    w2 = w2 - eta * g2
    path.append((w1, w2))

path = np.array(path)

### Visualization: contour plot with gradient descent path

In [ ]:
W1 = np.linspace(-0.5, 2.5, 100)
W2 = np.linspace(-0.5, 2.5, 100)
W1g, W2g = np.meshgrid(W1, W2)
L = loss_2d(W1g, W2g)

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
ax.contour(W1g, W2g, L, levels=15)
ax.plot(path[:, 0], path[:, 1], 'ro-', markersize=4, label='descent path')
ax.plot(1, 1, 'g*', markersize=14, label='minimum (1, 1)')
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Gradient descent on a 2D loss surface')
ax.legend()
ax.set_aspect('equal')
plt.tight_layout()
plt.show()

---
## 4. Backpropagation: how the network gets its gradients

### Intuition

A deep network has many parameters. **Backpropagation** computes the gradient of the loss with respect to every parameter in a single backward pass:

- The loss is a function of the output; the output is a function of the last layer; the last layer is a function of the one before it; and so on.
- The **chain rule** allows us to combine the gradient arriving from the layer above with the local derivative at the current layer, then pass the result further back.
- Starting from the loss, each layer receives a gradient signal, applies its local derivative, and forwards the result to the previous layer.

The outcome is a gradient for every weight and bias — a number indicating how much the loss would change for a small increase in that parameter. Parameters are then updated with a standard gradient step.

### Demo: gradients flowing backward

A minimal 2-layer network in NumPy. One forward pass computes the loss; the manual backward pass applies the chain rule at each layer. Gradients are printed and then plotted by layer so the backward flow is visible.

In [ ]:
np.random.seed(42)

# Tiny network: input 2 -> hidden 3 -> output 1
# Layer 1: (2, 3), Layer 2: (3, 1)
W1 = np.random.randn(2, 3) * 0.5
b1 = np.zeros(3)
W2 = np.random.randn(3, 1) * 0.5
b2 = np.zeros(1)

x = np.array([[1.0, 0.5]])  # one sample, 2 features
y_true = np.array([[2.0]])  # target

# --- Forward pass ---
z1 = x @ W1 + b1  # @ is Python's matrix multiplication operator (equivalent to np.dot)
a1 = np.maximum(0, z1)  # ReLU
z2 = a1 @ W2 + b2
y_pred = z2  # linear output (regression)

loss = np.mean((y_pred - y_true) ** 2)
print("Forward: loss =", loss)

# --- Backward pass (chain rule) ---
# dL/dy_pred = 2*(y_pred - y_true) / n
n = 1
dL_dy = 2 * (y_pred - y_true) / n
# Output layer: dL/dz2 = dL_dy (linear)
dL_dz2 = dL_dy
dL_dW2 = a1.T @ dL_dz2
dL_db2 = np.sum(dL_dz2, axis=0)
# Propagate to hidden: dL/da1
dL_da1 = dL_dz2 @ W2.T
# ReLU: gradient only where z1 > 0
dL_dz1 = dL_da1 * (z1 > 0)
dL_dW1 = x.T @ dL_dz1
dL_db1 = np.sum(dL_dz1, axis=0)

print("\nGradients (backprop):")
print("  dL/dW2 (output layer):", dL_dW2.flatten())
print("  dL/db2:", dL_db2)
print("  dL/dW1 (hidden layer):", dL_dW1.flatten())
print("  dL/db1:", dL_db1)

### Visualization: gradient magnitudes per layer

In [ ]:
# Gradient norms per parameter tensor (so we "see" flow backward)
grad_norms = [np.linalg.norm(dL_dW2), np.linalg.norm(dL_db2), np.linalg.norm(dL_dW1), np.linalg.norm(dL_db1)]
labels = ['dW2', 'db2', 'dW1', 'db1']

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.barh(labels[::-1], grad_norms[::-1], color=['steelblue', 'steelblue', 'coral', 'coral'])
ax.set_xlabel('Gradient norm')
ax.set_title('Gradients flowing backward: output layer (dW2, db2) → hidden (dW1, db1)')
plt.tight_layout()
plt.show()

### Reflection

**Summarise in one sentence: what does a backward pass compute?**

*Answer:* It applies the chain rule from the loss backward through each layer to produce a gradient for every parameter — the rate of change of the loss with respect to that parameter.

---
## 5. Guided coding: loss and one gradient step

Tasks:
1. Implement mean squared error for a given prediction and target.
2. Carry out one gradient descent step on a 1D model (one weight).

**Core path:** Complete the TODOs. **Extension:** Derive the gradient of MSE by hand — differentiate $\frac{1}{n}\sum (\hat{y}_i - y_i)^2$ with respect to $w$ — then verify it matches the expression already used in the code.

### Coding Exercise: Mean Squared Error

⏱ **Suggested time:** 5–8 minutes

**Instructions:**
1. Implement the `mse` function below.
2. If you are stuck for more than 3 minutes, open the solution.
3. If you finish early: what would the MSE be if all predictions were exactly equal to their targets?

**Goal:** Understand how MSE reduces prediction error to a single scalar that training can minimise.

In [ ]:
def mse(y_pred, y_true):
    """Mean squared error"""
    # TODO: compute the mean squared error
    pass

# Test
y_pred = np.array([1.0, 2.0, 3.0])
y_true = np.array([1.1, 2.2, 2.8])
print("MSE:", mse(y_pred, y_true))

<details>
<summary>Show solution</summary>

```python
def mse(y_pred, y_true):
    squared_errors = (y_pred - y_true) ** 2
    return np.mean(squared_errors)
```

</details>

### Coding Exercise: One Gradient Step

⏱ **Suggested time:** 10–12 minutes

**Instructions:**
1. Implement the three steps inside `one_gradient_step_1d`.
2. If you are stuck for more than 5 minutes, open the solution.
3. If you finish early, try the extension task below.

**Goal:** Understand how a single gradient descent update changes the parameter — and watch the loss decrease step by step.

---

**Model:** $\hat{y} = w \cdot x$ (one weight $w$, one input $x$).  
**Loss** for one sample: $L = (wx - y)^2$.  
**Gradient** with respect to $w$: $\dfrac{\partial L}{\partial w} = 2(wx - y) \cdot x$.  
**Update rule:** $w \leftarrow w - \eta \cdot \dfrac{\partial L}{\partial w}$.

In [ ]:
def one_gradient_step_1d(w, x, y, eta=0.1):
    """One gradient descent step for y_pred ."""
    # TODO: compute prediction y_pred 
    # TODO: compute gradient 
    # TODO: update and return w_new 
    # return w_new
   

# Example: fit w so that w * 2 ≈ 5 (target y=5, x=2)
w = 0.0
x, y = 2.0, 5.0
for step in range(10):
    loss_val = (w * x - y) ** 2
    print(f"Step {step}: w = {w:.4f}, loss = {loss_val:.4f}")
    w = one_gradient_step_1d(w, x, y, eta=0.1)
print(f"After 10 steps: w = {w:.4f} (expected about 2.5)")

<details>
<summary>Show solution</summary>

```python
def one_gradient_step_1d(w, x, y, eta=0.1):
    y_pred = w * x
    gradient = 2 * (y_pred - y) * x
    w_new = w - eta * gradient
    return w_new
```

</details>

**EXTENSION:** Derive the gradient analytically: differentiate $L = (w x - y)^2$ with respect to $w$, implement it, and verify it matches the expression already in the code above.

### Optional: plot loss over steps

In [ ]:
w = 0.0
x, y = 2.0, 5.0
eta = 0.1
losses = []
for step in range(20):
    loss_val = (w * x - y) ** 2
    losses.append(loss_val)
    w = one_gradient_step_1d(w, x, y, eta=eta)

plt.figure(figsize=(6, 3))
plt.plot(losses, 'o-')
plt.xlabel('Step')
plt.ylabel('Loss')
plt.title('Loss during gradient descent (1D model)')
plt.tight_layout()
plt.show()

---
## 6. Exploration: learning rate experiments and wrap-up

Run a training loop on a small 2-layer network and study the effect of learning rate. Try at least two or three values (e.g. 0.01, 0.1, 0.5) and compare the resulting loss curves.

**Core path:** Change `learning_rate`, re-run the training cell, and observe convergence speed and stability. **Extension:** Implement a simple momentum update, or investigate the effect of different weight initialisations.

## Exploration Block

⏱ **Suggested time:** 20 minutes

**Core task:**
- Run the training loop below.
- Try 2–3 different learning rates (e.g. `0.01`, `0.1`, `0.5`) and observe how the loss curve changes.

**If you finish early:**
- Try changing the number of training steps (`n_epochs`).
- Try replacing the MSE loss with mean absolute error (`np.mean(np.abs(y_pred - y))`) and see how training behaviour changes.

### TODO: Training loop with configurable learning rate

The cell below defines a small MLP and a full training loop. Set `learning_rate` to different values (e.g. `0.01`, `0.1`, `0.5`), re-run, and observe how the loss curve changes. Then use the comparison cell below to overlay several learning rates in a single plot.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def forward(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = relu(Z1)
    Z2 = A1 @ W2 + b2
    return Z2, A1, Z1

def backward(X, A1, Z1, W2, y_pred, y_true):
    n = X.shape[0]
    dL_dy = 2 * (y_pred - y_true) / n
    dL_dz2 = dL_dy
    dL_dW2 = A1.T @ dL_dz2
    dL_db2 = np.sum(dL_dz2, axis=0)
    dL_da1 = dL_dz2 @ W2.T
    dL_dz1 = dL_da1 * (Z1 > 0)
    dL_dW1 = X.T @ dL_dz1
    dL_db1 = np.sum(dL_dz1, axis=0)
    return dL_dW1, dL_db1, dL_dW2, dL_db2

# Toy data: 4 samples, 2 features
X = np.array([[1., 0.5], [0.5, 1.], [0.3, 0.8], [0.8, 0.3]], dtype=np.float64)
y = np.array([[2.0], [1.5], [1.2], [1.8]], dtype=np.float64)

# TODO: try different learning rates, e.g. 0.01, 0.1, 0.5
learning_rate = 0.1

np.random.seed(42)
W1 = np.random.randn(2, 4) * 0.5
b1 = np.zeros(4)
W2 = np.random.randn(4, 1) * 0.5
b2 = np.zeros(1)

n_epochs = 80
losses = []

for epoch in range(n_epochs):
    y_pred, A1, Z1 = forward(X, W1, b1, W2, b2)
    loss = np.mean((y_pred - y) ** 2)
    losses.append(loss)
    dW1, db1, dW2, db2 = backward(X, A1, Z1, W2, y_pred, y)
    W1 -= learning_rate * dW1
    b1 -= learning_rate * db1
    W2 -= learning_rate * dW2
    b2 -= learning_rate * db2

print(f"Final loss (lr={learning_rate}): {losses[-1]:.6f}")
plt.figure(figsize=(6, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title(f'Loss curve (learning_rate = {learning_rate})')
plt.tight_layout()
plt.show()

### Learning rate comparison

Run the cell below to compare loss curves for several learning rates. **Interpret:** too small → slow convergence; too large → instability or divergence.

In [ ]:
def train_and_return_losses(X, y, learning_rate, n_epochs=80, seed=42):
    np.random.seed(seed)
    W1 = np.random.randn(2, 4) * 0.5
    b1 = np.zeros(4)
    W2 = np.random.randn(4, 1) * 0.5
    b2 = np.zeros(1)
    losses = []
    for epoch in range(n_epochs):
        y_pred, A1, Z1 = forward(X, W1, b1, W2, b2)
        loss = np.mean((y_pred - y) ** 2)
        losses.append(loss)
        dW1, db1, dW2, db2 = backward(X, A1, Z1, W2, y_pred, y)
        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1
        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2
    return losses

learning_rates = [0.01, 0.1, 0.3, 0.5]
plt.figure(figsize=(7, 4))
for lr in learning_rates:
    ls = train_and_return_losses(X, y, lr)
    plt.plot(ls, label=f'lr={lr}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Learning rate comparison')
plt.legend()
plt.tight_layout()
plt.show()

For post-class extension tasks (momentum, early stopping, learning rate decay, cosine schedule) see the **Post-class Extensions** section at the end of this notebook.

### Wrap-up: takeaways

1. **Loss** measures how wrong the model is (e.g. MSE for regression). We minimise it.
2. **Gradient** points in the direction of steepest increase of the loss; we step *opposite* (gradient descent).
3. **One training step:** forward pass → compute loss → compute gradients (backprop) → update parameters.
4. **Backpropagation** uses the chain rule so every parameter gets a gradient; gradients "flow" backward through the network.
5. **Learning rate** controls step size: too small → slow; too large → unstable. Experiment to find a good range.

**What's next:** Optimizers (e.g. momentum, Adam), regularization (dropout, batch norm), and deeper architectures.

---

## Post-class Extensions

*Work on these after class, or during the session if you finish the main exercises early.*

Each exercise improves the training loop by adding one feature. The final cell plots all variants together so you can see the **cumulative benefit** of each addition.

| Exercise | Feature added |
|----------|--------------|
| A | Momentum |
| B | Early stopping with patience |
| C | Learning rate decay |
| D *(advanced)* | Cosine learning rate schedule |

All exercises use the same toy dataset, `forward`, and `backward` functions from Section 6.

### A. Momentum

**Goal:** Smooth parameter updates by accumulating a running velocity. This reduces oscillations and often converges faster than plain gradient descent.

**How it works:**
```
v = momentum * v + grad
w -= lr * v
```

Try `momentum = 0.5`, `0.9`, `0.99` and observe how the loss curve changes.

In [ ]:
N_EPOCHS = 200  # shared epoch count for all extension exercises

def train_momentum(X, y, lr=0.1, momentum=0.9, n_epochs=N_EPOCHS, seed=42):
    np.random.seed(seed)
    W1 = np.random.randn(2, 4) * 0.5; b1 = np.zeros(4)
    W2 = np.random.randn(4, 1) * 0.5; b2 = np.zeros(1)
    vW1, vb1 = np.zeros_like(W1), np.zeros_like(b1)
    vW2, vb2 = np.zeros_like(W2), np.zeros_like(b2)
    losses = []
    for _ in range(n_epochs):
        y_pred, A1, Z1 = forward(X, W1, b1, W2, b2)
        losses.append(np.mean((y_pred - y) ** 2))
        dW1, db1, dW2, db2 = backward(X, A1, Z1, W2, y_pred, y)
        vW1 = momentum * vW1 + dW1; vb1 = momentum * vb1 + db1
        vW2 = momentum * vW2 + dW2; vb2 = momentum * vb2 + db2
        W1 -= lr * vW1; b1 -= lr * vb1
        W2 -= lr * vW2; b2 -= lr * vb2
    return losses

baseline      = train_and_return_losses(X, y, learning_rate=0.1, n_epochs=N_EPOCHS)
with_momentum = train_momentum(X, y, lr=0.1, momentum=0.9)

plt.figure(figsize=(7, 3))
plt.plot(baseline,      label='Baseline (GD)')
plt.plot(with_momentum, label='+ Momentum (0.9)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('A. Effect of momentum')
plt.legend(); plt.tight_layout(); plt.show()

### B. Early stopping with patience

**Goal:** Stop training automatically when the loss stops improving, saving time and reducing risk of overfitting.

**How it works:** Count consecutive epochs with no improvement. When that count reaches `patience_epochs`, stop.

The curve is padded to `N_EPOCHS` for a fair visual comparison — the flat tail shows where training was halted.

Try `patience_epochs = 5`, `10`, `20`.

In [ ]:
def train_early_stopping(X, y, lr=0.1, patience_epochs=10, n_epochs=N_EPOCHS, seed=42):
    np.random.seed(seed)
    W1 = np.random.randn(2, 4) * 0.5; b1 = np.zeros(4)
    W2 = np.random.randn(4, 1) * 0.5; b2 = np.zeros(1)
    losses = []; best_loss = np.inf; no_improve = 0
    for epoch in range(n_epochs):
        y_pred, A1, Z1 = forward(X, W1, b1, W2, b2)
        loss = np.mean((y_pred - y) ** 2); losses.append(loss)
        if loss < best_loss:
            best_loss = loss; no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience_epochs:
                print(f"Early stop at epoch {epoch} (patience={patience_epochs})")
                break
        dW1, db1, dW2, db2 = backward(X, A1, Z1, W2, y_pred, y)
        W1 -= lr * dW1; b1 -= lr * db1
        W2 -= lr * dW2; b2 -= lr * db2
    return losses

with_patience = train_early_stopping(X, y, lr=0.1, patience_epochs=10)
# Pad to N_EPOCHS so the flat tail shows where training stopped
padded_patience = with_patience + [with_patience[-1]] * (N_EPOCHS - len(with_patience))

plt.figure(figsize=(7, 3))
plt.plot(baseline,        label='Baseline (GD)')
plt.plot(padded_patience, label='+ Early stopping (patience=10)', linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('B. Effect of early stopping')
plt.legend(); plt.tight_layout(); plt.show()

### C. Learning rate decay

**Goal:** Start with a larger step size for fast initial progress, then shrink it so the model settles more precisely near the minimum.

**How it works:** Multiply the current learning rate by a decay factor after each epoch:
```
lr = lr * decay
```

Try `decay = 0.99`, `0.95`, `0.90`. Lower decay means faster shrinkage.

In [ ]:
def train_lr_decay(X, y, lr=0.1, decay=0.99, n_epochs=N_EPOCHS, seed=42):
    np.random.seed(seed)
    W1 = np.random.randn(2, 4) * 0.5; b1 = np.zeros(4)
    W2 = np.random.randn(4, 1) * 0.5; b2 = np.zeros(1)
    losses = []; current_lr = lr
    for _ in range(n_epochs):
        y_pred, A1, Z1 = forward(X, W1, b1, W2, b2)
        losses.append(np.mean((y_pred - y) ** 2))
        dW1, db1, dW2, db2 = backward(X, A1, Z1, W2, y_pred, y)
        W1 -= current_lr * dW1; b1 -= current_lr * db1
        W2 -= current_lr * dW2; b2 -= current_lr * db2
        current_lr *= decay
    return losses

with_decay = train_lr_decay(X, y, lr=0.1, decay=0.99)

plt.figure(figsize=(7, 3))
plt.plot(baseline,   label='Baseline (GD)')
plt.plot(with_decay, label='+ LR decay (×0.99/epoch)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('C. Effect of learning rate decay')
plt.legend(); plt.tight_layout(); plt.show()

### D. Cosine learning rate schedule *(advanced)*

**Goal:** Use a smooth cosine curve to anneal the learning rate from `lr_max` down to `lr_min`. This is widely used in modern training recipes (e.g. ResNet, ViT).

**Formula:**

$$lr_t = lr_{min} + \frac{1}{2}(lr_{max} - lr_{min})\left(1 + \cos\!\left(\frac{\pi \cdot t}{T}\right)\right)$$

The right subplot shows the schedule itself — notice how it starts high, drops smoothly, and ends near zero.

In [ ]:
def train_cosine_schedule(X, y, lr_max=0.3, lr_min=0.001, n_epochs=N_EPOCHS, seed=42):
    np.random.seed(seed)
    W1 = np.random.randn(2, 4) * 0.5; b1 = np.zeros(4)
    W2 = np.random.randn(4, 1) * 0.5; b2 = np.zeros(1)
    losses = []; lr_schedule = []
    for epoch in range(n_epochs):
        lr = lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * epoch / n_epochs))
        lr_schedule.append(lr)
        y_pred, A1, Z1 = forward(X, W1, b1, W2, b2)
        losses.append(np.mean((y_pred - y) ** 2))
        dW1, db1, dW2, db2 = backward(X, A1, Z1, W2, y_pred, y)
        W1 -= lr * dW1; b1 -= lr * db1
        W2 -= lr * dW2; b2 -= lr * db2
    return losses, lr_schedule

with_cosine, cosine_lr = train_cosine_schedule(X, y)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
ax1.plot(baseline,    label='Baseline')
ax1.plot(with_cosine, label='Cosine schedule')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('D. Loss: cosine vs baseline'); ax1.legend()
ax2.plot(cosine_lr, color='orange')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Learning rate')
ax2.set_title('Cosine LR schedule')
plt.tight_layout(); plt.show()

### Cumulative comparison

All five variants on one plot. Each curve adds one improvement over the plain baseline, so you can read off the **marginal benefit** of each addition.

| Curve | What is added |
|-------|--------------|
| 1. Baseline | Plain gradient descent |
| 2. + Momentum | Running velocity smooths updates |
| 3. + Early stopping | Halts when loss plateaus (flat tail = stopped early) |
| 4. + LR decay | Step size shrinks each epoch |
| 5. + Cosine schedule | Smooth lr annealing *(advanced)* |

In [ ]:
plt.figure(figsize=(9, 4))
plt.plot(baseline,        label='1. Baseline (GD, lr=0.1)')
plt.plot(with_momentum,   label='2. + Momentum (0.9)')
plt.plot(padded_patience, label='3. + Early stopping (patience=10)', linestyle='--')
plt.plot(with_decay,      label='4. + LR decay (×0.99/epoch)')
plt.plot(with_cosine,     label='5. + Cosine schedule (advanced)')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.title('Cumulative effect of training loop improvements')
plt.legend(loc='upper right'); plt.tight_layout(); plt.show()